# Assessment Task No. 6 — SMS Spam Classification with Multinomial Naive Bayes
**CSST102 – Basic Machine Learning | Laguna State Polytechnic University**

Run all cells top to bottom (Runtime ▸ Run all). When asked, upload `spam.csv` (it is inside your `implementation` folder).

## Setup

In [ ]:
import re, string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)
pd.set_option("display.max_colwidth", 90)

# Load spam.csv (upload it in Colab if it is not found)
import os
if not os.path.exists("spam.csv"):
    from google.colab import files
    files.upload()

## Task 1 — Dataset Exploration and Analysis

In [ ]:
df = pd.read_csv("spam.csv", encoding="latin-1")
print("Dimensions (rows, columns):", df.shape)
df.head(10)

In [ ]:
print("Missing values per column:\n", df.isnull().sum())
print("\nDuplicate messages (v2):", df["v2"].duplicated().sum())

In [ ]:
# Remove unnecessary columns, rename v1 -> class, v2 -> message
df = df[["v1", "v2"]].rename(columns={"v1": "class", "v2": "message"})
dups = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)      # remove duplicate rows
print("Exact duplicate rows removed:", dups)
print("Cleaned dataset shape:", df.shape)

counts = df["class"].value_counts()
summary = pd.DataFrame({"Count": counts, "Percentage (%)": (counts / len(df) * 100).round(2)})
print(summary)
df.describe()

In [ ]:
ax = counts.plot(kind="bar", color=["#4C72B0", "#C44E52"], rot=0, figsize=(6, 4))
for i, v in enumerate(counts): ax.text(i, v + 30, f"{v} ({v/len(df)*100:.1f}%)", ha="center")
plt.title("Class Distribution of SMS Messages"); plt.xlabel("Class"); plt.ylabel("Number of messages")
plt.tight_layout(); plt.show()

**Interpretation:** the dataset is **imbalanced** — ham messages are the large majority and spam is the minority class. Accuracy alone can therefore be misleading, so precision, recall and F1 for the spam class are also used.

## Task 2 — NLP Text Preprocessing
Steps: lowercase → remove punctuation/special characters → remove digits → tokenize → remove English stopwords → Porter stemming. (The stopword list is NLTK's standard English list and the stemmer is the Porter algorithm, both embedded so no downloads are needed.)

In [ ]:
# Compact implementation of the classic Porter (1980) stemming algorithm
import re
class PorterStemmer:
    def _cons(self, w, i):
        c = w[i]
        if c in 'aeiou': return False
        if c == 'y': return i == 0 or not self._cons(w, i-1)
        return True
    def _m(self, s):
        form = ''.join('c' if self._cons(s, i) else 'v' for i in range(len(s)))
        form = re.sub(r'(.)\1+', r'\1', form)
        return form.count('vc')
    def _hasv(self, s): return any(not self._cons(s, i) for i in range(len(s)))
    def _dc(self, w): return len(w) > 1 and w[-1] == w[-2] and self._cons(w, len(w)-1)
    def _cvc(self, w):
        return (len(w) >= 3 and self._cons(w, len(w)-3) and not self._cons(w, len(w)-2)
                and self._cons(w, len(w)-1) and w[-1] not in 'wxy')
    def _rep(self, w, rules, mm):
        for suf, rep in rules:
            if w.endswith(suf):
                stem = w[:-len(suf)]
                return stem + rep if self._m(stem) > mm else w
        return w
    def stem(self, w):
        if len(w) <= 2: return w
        # step 1a
        if w.endswith('sses'): w = w[:-2]
        elif w.endswith('ies'): w = w[:-2]
        elif w.endswith('ss'): pass
        elif w.endswith('s'): w = w[:-1]
        # step 1b
        flag = False
        if w.endswith('eed'):
            if self._m(w[:-3]) > 0: w = w[:-1]
        else:
            for suf in ('ed', 'ing'):
                if w.endswith(suf) and self._hasv(w[:-len(suf)]):
                    w = w[:-len(suf)]; flag = True; break
            if flag:
                if w.endswith(('at', 'bl', 'iz')): w += 'e'
                elif self._dc(w) and w[-1] not in 'lsz': w = w[:-1]
                elif self._m(w) == 1 and self._cvc(w): w += 'e'
        # step 1c
        if w.endswith('y') and self._hasv(w[:-1]): w = w[:-1] + 'i'
        # step 2
        w = self._rep(w, [('ational','ate'),('tional','tion'),('enci','ence'),('anci','ance'),('izer','ize'),
            ('abli','able'),('alli','al'),('entli','ent'),('eli','e'),('ousli','ous'),('ization','ize'),
            ('ation','ate'),('ator','ate'),('alism','al'),('iveness','ive'),('fulness','ful'),
            ('ousness','ous'),('aliti','al'),('iviti','ive'),('biliti','ble')], 0)
        # step 3
        w = self._rep(w, [('icate','ic'),('ative',''),('alize','al'),('iciti','ic'),('ical','ic'),
            ('ful',''),('ness','')], 0)
        # step 4
        for suf in ['al','ance','ence','er','ic','able','ible','ant','ement','ment','ent','ion','ou',
                    'ism','ate','iti','ous','ive','ize']:
            if w.endswith(suf):
                stem = w[:-len(suf)]
                if self._m(stem) > 1 and (suf != 'ion' or stem[-1:] in ('s','t')): w = stem
                break
        # step 5
        if w.endswith('e'):
            stem = w[:-1]; m = self._m(stem)
            if m > 1 or (m == 1 and not self._cvc(stem)): w = stem
        if self._m(w) > 1 and self._dc(w) and w.endswith('l'): w = w[:-1]
        return w

In [ ]:
STOPWORDS = set("""i me my myself we our ours ourselves you you're you've you'll you'd your yours yourself yourselves he him his himself she she's her hers herself it it's its itself they them their theirs themselves what which who whom this that that'll these those am is are was were be been being have has had having do does did doing a an the and but if or because as until while of at by for with about against between into through during before after above below to from up down in out on off over under again further then once here there when where why how all any both each few more most other some such no nor not only own same so than too very s t can will just don don't should should've now d ll m o re ve y ain aren aren't couldn couldn't didn didn't doesn doesn't hadn hadn't hasn hasn't haven haven't isn isn't ma mightn mightn't mustn mustn't needn needn't shan shan't shouldn shouldn't wasn wasn't weren weren't won won't wouldn wouldn't""".split())
stemmer = PorterStemmer()

def preprocess(text):
    text = text.lower()                                              # 1. lowercase
    text = text.translate(str.maketrans("", "", string.punctuation)) # 2. remove punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)                         #    remove special characters
    text = re.sub(r"\d+", "", text)                                  # 3. remove digits
    tokens = text.split()                                            # 4. tokenization
    tokens = [t for t in tokens if t not in STOPWORDS]               # 5. stopword removal
    tokens = [stemmer.stem(t) for t in tokens]                       # 6. stemming
    return " ".join(tokens)

df["clean"] = df["message"].apply(preprocess)
df["label"] = (df["class"] == "spam").astype(int)     # spam = 1, ham = 0
df[["message", "clean"]].head(5)

**Analysis:** Preprocessing reduces vocabulary size and noise. Lowercasing, stemming and digit/punctuation removal merge variants (e.g., *win, wins, winning* → *win*), so word counts are less sparse and Naive Bayes can estimate P(word | class) more reliably. Removing stopwords removes high-frequency words that carry little class information. Because Naive Bayes assumes words are independent and relies purely on word counts, cleaner input generally gives better probability estimates, though over-aggressive cleaning (e.g., removing digits) can also discard useful spam cues such as prices or phone numbers.

## Task 3 — Text Feature Extraction

In [ ]:
cv_all = CountVectorizer();  X_cv_all = cv_all.fit_transform(df["clean"])
tf_all = TfidfVectorizer();  X_tf_all = tf_all.fit_transform(df["clean"])

for name, vec, X in [("CountVectorizer", cv_all, X_cv_all), ("TfidfVectorizer", tf_all, X_tf_all)]:
    print(f"--- {name} ---")
    print("Feature matrix dimensions:", X.shape)
    print("Vocabulary size:", len(vec.vocabulary_))
    print("10 example features:", list(vec.get_feature_names_out()[1000:1010]))
    print()

**Analysis:** CountVectorizer uses *word frequency* — the raw number of times each term appears in a message. TF-IDF multiplies term frequency by inverse document frequency, so words that appear in many messages are down-weighted and rare, distinctive words get higher weights (and rows are length-normalised). For spam classification, **TF-IDF** is often more suitable because it reduces the influence of common words and emphasizes discriminative terms like *prize*, *claim* or *urgent*. Which one actually performs better is tested empirically in Task 5.

## Task 4 — Naive Bayes Model Development
70/30 stratified split, `random_state=42`; vectorizers are fitted on the **training data only** (no data leakage).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=0.30, stratify=df["label"], random_state=42)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

# Model A: CountVectorizer + MultinomialNB
cv = CountVectorizer()
Xtr_a = cv.fit_transform(X_train); Xte_a = cv.transform(X_test)
model_a = MultinomialNB().fit(Xtr_a, y_train)
pred_a = model_a.predict(Xte_a)

# Model B: TfidfVectorizer + MultinomialNB
tfidf = TfidfVectorizer()
Xtr_b = tfidf.fit_transform(X_train); Xte_b = tfidf.transform(X_test)
model_b = MultinomialNB().fit(Xtr_b, y_train)
pred_b = model_b.predict(Xte_b)

pred_df = pd.DataFrame({"message": X_test.values, "actual": y_test.map({0:"ham",1:"spam"}).values,
                        "Model A": pd.Series(pred_a).map({0:"ham",1:"spam"}).values,
                        "Model B": pd.Series(pred_b).map({0:"ham",1:"spam"}).values})
pred_df.head(10)

## Task 5 — Model Evaluation and Comparison

In [ ]:
def evaluate(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred),
            "Recall": recall_score(y_true, y_pred),
            "F1-score": f1_score(y_true, y_pred),
            "False Positives": fp, "False Negatives": fn}

results = pd.DataFrame({"Model A": evaluate(y_test, pred_a), "Model B": evaluate(y_test, pred_b)})
results_display = results.copy()
results_display.loc[["Accuracy","Precision","Recall","F1-score"]] = results.loc[["Accuracy","Precision","Recall","F1-score"]].astype(float).round(4)
print(results_display)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, pred, title in zip(axes, [pred_a, pred_b], ["Model A (CountVectorizer)", "Model B (TF-IDF)"]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["ham","spam"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(title)
plt.tight_layout(); plt.show()

**Analysis:** **Model A (CountVectorizer) performs better overall** — higher accuracy (97.94% vs 95.87%), recall (88.27% vs 67.86%) and F1 (0.9153 vs 0.8061). Model B has slightly higher precision (99.25% vs 95.05%; 1 vs 9 false positives) but misses far more spam (63 vs 23 false negatives). Precision (of the messages flagged as spam, how many really are) matters because a false positive sends a legitimate message to the spam folder, where the user may never see it. Recall (of all real spam, how many were caught) matters because a false negative lets spam through. A good filter needs both high; F1 balances them, and in spam filtering false positives are usually considered the costlier error.

## Task 6 — Testing with Unseen Messages

In [ ]:
# Pick the better model by F1-score (ties broken by precision)
use_b = (results.loc["F1-score","Model B"], results.loc["Precision","Model B"]) > (results.loc["F1-score","Model A"], results.loc["Precision","Model A"])
best_name = "Model B (TF-IDF)" if use_b else "Model A (CountVectorizer)"
best_vec, best_model = (tfidf, model_b) if use_b else (cv, model_a)
print("Better-performing model:", best_name, "\n")

new_msgs = ["Congratulations! You have won a free cash prize.",
            "Good morning, our class starts at 8 AM tomorrow.",
            "Claim your reward now by clicking this link.",
            "Please submit your assignment before Friday.",
            "URGENT! You have been selected to receive a cash bonus."]
X_new = best_vec.transform([preprocess(m) for m in new_msgs])
proba = best_model.predict_proba(X_new)
out = pd.DataFrame({"No.": range(1, 6), "SMS Message": new_msgs,
                    "Prediction": np.where(proba[:, 1] >= 0.5, "spam", "ham"),
                    "P(spam)": proba[:, 1].round(6), "P(ham)": proba[:, 0].round(6)})
out

**Analysis:** All five predictions match what a human would expect (1, 3, 5 = spam; 2, 4 = ham), so there are no outright misclassifications. Points worth noting: (a) message 4 (*Please submit your assignment before Friday.*) is ham but still gets P(spam) ≈ 0.057, because *please* and *submit* lean slightly spam in the training data and *assignment* is not in the vocabulary, so it is ignored; (b) message 3 has no money words and relies mostly on *claim*, yet gets P(spam) ≈ 0.9998; (c) probabilities are extremely close to 0 or 1 because Naive Bayes assumes words are independent and multiplies their probabilities, which makes it overconfident; (d) the model sees only words, not context or sender, so a legitimate message using words like *free*, *claim* or *prize* could be wrongly flagged.